# Report Label Extraction — RSNA Knee

Mines the twelve findings from the 4,349 training studies that have a radiology
report but no radiologist labels. Those mined labels are the training signal for
the imaging model; only 58 studies carry real labels, and those are reserved for
validation.

**Runtime:** A100 recommended. Roughly 45–75 min for the full corpus at 3
self-consistency samples per report.

**Data handling:** the model runs locally on this Colab GPU. Report text is
never sent to a hosted API.

### What you do

1. `Runtime → Change runtime type → A100 GPU`
2. Run every cell top to bottom.
3. Cell 6 is a checkpoint: it scores the extractor on the 58 gold studies and
   prints a reliability table. **Read it before running the full corpus** — if
   macro AUC is not comfortably above 0.70, stop and report the numbers back
   rather than spending an hour.
4. The last cell downloads `llm_labels.parquet`. Drop that into `data/meta/`.

## 1. Check the GPU and pick a model size

In [ ]:
import subprocess, torch

# nvidia-smi is absent on a CPU runtime, which is exactly the case the
# message below is for -- so this must not raise first.
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("nvidia-smi not found")

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 GPU.")

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {VRAM_GB:.0f} GB")

# AWQ 4-bit puts the 32B at ~19 GB of weights, so it fits an A100 40GB or an
# L4 24GB and replaces the 14B-bf16 that used to top this ladder. The gold
# checkpoint is the arbiter: compare its per-finding AUCs against the 14B run
# before spending an hour of corpus extraction on a bigger model.
if VRAM_GB >= 22:
    MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"
elif VRAM_GB >= 15:
    MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
else:
    MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

print("model:", MODEL)

## 2. Install vLLM

Batched generation over ~13,000 prompts (4,349 reports × 3 samples). vLLM's
continuous batching is roughly an order of magnitude faster than a plain
`transformers` loop here.

Colab prompts to restart the runtime after this. **Restart, then re-run
section 1, skip this section, and continue from section 3.** The restart is
required: vLLM replaces torch and the CUDA libraries underneath a process that
has already imported the old ones.

**The `ERROR: pip's dependency resolver ...` wall of text here is noise, and
the install it follows succeeded** — pip prints that report at ERROR level
after resolving, not instead of installing. Every conflict is against
packages this notebook never imports: RAPIDS (`cudf`, `cuml`, `raft`, `rmm`,
`cuvs`, `dask-cuda`, `numba-cuda`, and their `cuda-toolkit` / `cuda-python`
pins, which vLLM's newer CUDA wheels invalidate), the Google SDKs
(`google-adk`, `google-ai-generativelanguage`, `grpcio-status` over
protobuf), and `ipython`'s optional `jedi` — that last one costs tab
completion and `obj?` introspection, nothing this notebook runs.

The cell prints a banner with the installed vLLM version underneath the
wall. **If that banner is missing, the install genuinely failed** — that is
the one case to stop and report; a conflict list above a banner is not.

In [ ]:
!pip install -q vllm

# vLLM installs a PyTorch built against a newer CUDA than Colab's preinstalled
# torchaudio, and vLLM's import chain touches torchaudio -- which then raises
# "PyTorch and TorchAudio were compiled with different CUDA versions" at
# `from vllm import LLM`. Nothing here uses audio, so the mismatch is removed
# rather than reconciled.
!pip uninstall -y -q torchaudio

# pip's conflict report is long enough to push the restart instruction off
# screen, and it reads like a failure while being none -- so confirm the
# install positively and say so louder than the noise above. Reading the
# version from package metadata does not import vllm, which must not happen
# before the restart.
from importlib.metadata import PackageNotFoundError, version

try:
    installed = version("vllm")
except PackageNotFoundError:
    installed = None

print()
print("=" * 68)
if installed:
    print(f"  vLLM {installed} installed. Conflicts printed above are noise.")
    print("  RESTART THE RUNTIME NOW, then re-run section 1, skip this")
    print("  cell, and continue from section 3.")
else:
    print("  vLLM did NOT install. Stop here and report the pip output --")
    print("  the conflict list above is normally harmless, so a missing")
    print("  version means something else failed.")
print("=" * 68)

## 3. Load `train.csv`

Either mount Drive (if you put the file there) or upload it directly. It is
~5.7 MB.

In [ ]:
import os
import pandas as pd

TRAIN_CSV = None

# Try Drive first, then fall back to an upload prompt.
for candidate in ["/content/drive/MyDrive/knee/train.csv", "/content/train.csv"]:
    if os.path.exists(candidate):
        TRAIN_CSV = candidate
        break

if TRAIN_CSV is None:
    from google.colab import files
    print("Select train.csv ...")
    uploaded = files.upload()
    TRAIN_CSV = list(uploaded.keys())[0]

train = pd.read_csv(TRAIN_CSV)
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

gold = train[train["ACL"].notna()].reset_index(drop=True)
unlabelled = train[train["ACL"].isna()].reset_index(drop=True)
print(f"{len(train)} studies: {len(gold)} labelled, {len(unlabelled)} report-only")
assert len(gold) == 58, f"expected 58 gold studies, found {len(gold)}"

## 4. Extraction logic

Generated from `src/labels/llm_extract.py` in the repo — **do not edit here**,
edit the module and regenerate, or the notebook and the repo will disagree
about how responses are parsed.

In [ ]:
"""Prompt construction and response parsing for the LLM report extractor.

This module is deliberately free of GPU and model dependencies so it can be
unit-tested on a laptop. The notebook that runs the model embeds this file's
source directly, so the two cannot drift apart.

Contract matches src/labels/lexicon.py exactly -- `present` is 1, 0, or None,
and `severity` is a float or None -- so both extractors can be scored by the
same calibration code and fused without translation.
"""

import json
import re
from typing import Dict, List, Optional

# Findings, with the definition the model is asked to apply. Ground truth was
# adjudicated with explicit severity thresholds, so the definitions state what
# counts rather than leaving the model to guess a threshold.
FINDING_DEFINITIONS = {
    "ACL": "Anterior cruciate ligament tear, rupture, or significant injury. "
           "Not intact/normal ACL, not a mention of the PCL.",
    # Reverted to the wording that measured better. The sharper radiological
    # phrasing -- surface-reaching signal, explicit exclusions -- was reasoned
    # rather than tested, and cost 0.02 macro and 0.15 on MCL alone.
    "MCL": "Medial collateral ligament tear, rupture, or sprain.",
    "Medial Meniscus": "Tear of the medial (internal) meniscus. Degeneration "
                       "without a tear does not count.",
    "Lateral Meniscus": "Tear of the lateral (external) meniscus. Degeneration "
                        "without a tear does not count.",
    "Medial OA": "Osteoarthritis, cartilage loss, or chondral thinning in the "
                 "MEDIAL femorotibial compartment.",
    "Lateral OA": "Osteoarthritis, cartilage loss, or chondral thinning in the "
                  "LATERAL femorotibial compartment.",
    "PF OA": "Osteoarthritis, chondromalacia, or cartilage loss in the "
             "PATELLOFEMORAL compartment (patella or trochlea).",
    "Effusion": "Joint effusion or increased intra-articular fluid.",
    "Synovitis": "Synovitis or synovial thickening/enhancement.",
    "Baker's": "Baker's cyst or popliteal cyst.",
    "Contusion": "Bone contusion or bone marrow oedema (excluding oedema that "
                 "is purely degenerative subchondral change).",
    "Fracture": "Any fracture, including stress, avulsion, and insufficiency "
                "fractures.",
}

PRESENCE_VALUES = {"yes": 1, "no": 0, "unclear": None}

# Severity and certainty map onto the same 0-1 scale the lexicon produces, so
# fusion compares like with like.
SEVERITY_SCORES = {"severe": 1.0, "moderate": 0.85, "minimal": 0.35, "none": 0.0}
CERTAINTY_FACTORS = {"definite": 1.0, "probable": 0.7, "possible": 0.5}

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. Reports may be in any language, including English, "
    "Spanish, Turkish, German, Greek, Bulgarian, Croatian, Dutch, French, or "
    "Italian. Read the report in its original language. Do not translate it. "
    "Report only what the text states. Never infer a finding that the report "
    "does not describe."
)


def build_prompt(report: str) -> str:
    """The user turn: the report plus the schema the model must fill in."""
    definitions = "\n".join(
        f'  "{name}": {definition}' for name, definition in FINDING_DEFINITIONS.items()
    )

    return f"""Extract these twelve findings from the knee MRI report below.

FINDINGS:
{definitions}

For each finding return:
  "present": "yes" if the report describes it as present,
             "no" if the report explicitly states it is absent, normal, or intact,
             "unclear" if the report does not mention it at all.
  "severity": "severe", "moderate", "minimal", or "none".
             Use "minimal" for trace, mild, or slight findings.
  "certainty": "definite", "probable", or "possible".
             Use "possible" when the report hedges (suspicious for, cannot exclude).
  "evidence": the exact phrase from the report, in its original language, that
             supports your answer. Use "" when present is "unclear".

Critical: "unclear" and "no" are different. Use "no" ONLY when the report
positively asserts the structure is normal or the finding is absent. If the
report is simply silent about a finding, use "unclear".

REPORT:
\"\"\"
{report}
\"\"\"

Respond with ONLY a JSON object, no commentary, in exactly this form:
{{"ACL": {{"present": "...", "severity": "...", "certainty": "...", "evidence": "..."}}, ...}}
Include all twelve findings."""


def _empty_result() -> Dict[str, Dict]:
    return {
        name: {"present": None, "severity": None, "evidence": None}
        for name in FINDING_DEFINITIONS
    }


def _extract_json_block(text: str) -> Optional[str]:
    """Find the outermost JSON object in a response.

    Models wrap JSON in prose or fenced blocks often enough that locating the
    braces is more reliable than requiring a clean response.
    """
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1]

    return None


def _score(severity: str, certainty: str) -> float:
    base = SEVERITY_SCORES.get(str(severity).strip().lower(), 0.85)
    factor = CERTAINTY_FACTORS.get(str(certainty).strip().lower(), 1.0)
    return base * factor


def parse_response(text: str) -> Dict[str, Dict]:
    """Turn one model response into the extractor contract.

    Unparseable responses and missing findings resolve to None rather than a
    guess -- a malformed generation must not be read as a confident negative.
    """
    result = _empty_result()
    if not isinstance(text, str) or not text.strip():
        return result

    block = _extract_json_block(text)
    if block is None:
        return result

    try:
        payload = json.loads(block)
    except json.JSONDecodeError:
        # Trailing commas are the usual culprit and are cheap to repair.
        repaired = re.sub(r",\s*([}\]])", r"\1", block)
        try:
            payload = json.loads(repaired)
        except json.JSONDecodeError:
            return result

    if not isinstance(payload, dict):
        return result

    for name in FINDING_DEFINITIONS:
        entry = payload.get(name)
        if not isinstance(entry, dict):
            continue

        raw_present = str(entry.get("present", "")).strip().lower()
        if raw_present not in PRESENCE_VALUES:
            continue

        present = PRESENCE_VALUES[raw_present]
        if present is None:
            result[name] = {"present": None, "severity": None, "evidence": None}
            continue

        severity = (
            _score(entry.get("severity", "moderate"), entry.get("certainty", "definite"))
            if present == 1 else 0.0
        )
        evidence = entry.get("evidence") or None
        result[name] = {
            "present": present,
            "severity": severity,
            "evidence": str(evidence)[:300] if evidence else None,
        }

    return result


def aggregate_samples(samples: List[Dict[str, Dict]]) -> Dict[str, Dict]:
    """Combine self-consistency samples into one result per finding.

    Majority vote decides presence; the vote fraction becomes the confidence
    that fusion uses to weight the sample. Findings where the model disagrees
    with itself are exactly the ones that should carry less weight.
    """
    if not samples:
        return {
            name: {"present": None, "severity": None, "evidence": None,
                   "confidence": 0.0}
            for name in FINDING_DEFINITIONS
        }

    aggregated = {}
    for name in FINDING_DEFINITIONS:
        votes = [s[name]["present"] for s in samples if name in s]
        counts = {
            1: votes.count(1),
            0: votes.count(0),
            None: votes.count(None),
        }
        winner = max(counts, key=lambda key: counts[key])
        confidence = counts[winner] / len(votes) if votes else 0.0

        matching = [
            s[name] for s in samples
            if name in s and s[name]["present"] == winner
        ]
        severities = [m["severity"] for m in matching if m["severity"] is not None]
        evidences = [m["evidence"] for m in matching if m["evidence"]]

        aggregated[name] = {
            "present": winner,
            "severity": (sum(severities) / len(severities)) if severities else None,
            "evidence": evidences[0] if evidences else None,
            "confidence": confidence,
        }

    return aggregated


def extract_finding_from_record(record: Dict, finding: str) -> Dict:
    """Adapter so calibrate.score_extractor can score cached LLM output."""
    entry = record.get(finding, {})
    return {
        "present": entry.get("present"),
        "severity": entry.get("severity"),
        "span": entry.get("evidence"),
    }


# ---------------------------------------------------------------------------
# Probability extraction
# ---------------------------------------------------------------------------
# The categorical contract above collapses 58 gold studies onto ~10-20 distinct
# target values, and 53% of all label slots share a single one. AUC is a
# ranking metric: a target that cannot order the studies inside a tie block
# gives the model nothing to learn there, and caps the achievable score.
#
# Asking for a probability instead produces a continuous score per study, which
# breaks the ties -- including for unmentioned findings, where the model can
# still reason from context ("severe tricompartmental OA" makes an unmentioned
# effusion more likely than a report describing an otherwise normal knee).

PROBABILITY_PROMPT = """Read the knee MRI report and estimate, for each of the
twelve findings, the probability that a musculoskeletal radiologist reviewing
the images would mark it present.

FINDINGS:
{definitions}

Give a probability from 0 to 100 for each.

How to think about it:
- Explicitly described as present: high, 85-99, scaled by how definite and how
  severe the description is. A "large joint effusion" is higher than a "trace
  effusion", because the reference standard applies severity thresholds.
- Explicitly described as absent, normal, or intact: low, 1-10.
- Hedged ("possible", "suspicious for", "cannot exclude"): middle, 30-60.
- Not mentioned at all: this is NOT the same as absent. Reports state positives
  and omit much else, and what gets omitted varies by finding and by
  radiologist. Estimate from the rest of the report and from how routinely that
  finding is reported. A report describing severe degenerative change makes an
  unmentioned effusion more likely; a detailed normal study makes it less so.
  Most unmentioned findings land between 5 and 35.

Do not round to 0, 50, or 100. Distinguishing two studies that both fail to
mention a finding is exactly what this is for.

REPORT:
\"\"\"
{report}
\"\"\"

Respond with ONLY a JSON object mapping each finding to a number:
{{"ACL": 3, "MCL": 12, "Medial Meniscus": 88, ...}}
Include all twelve findings."""


# ---------------------------------------------------------------------------
# Prompt variants (Phase C — extraction v3)
# ---------------------------------------------------------------------------
# Each candidate changes ONE idea against the baseline, so a gold-sweep delta
# is attributable to it. Variants are built by anchored substitution on the
# baseline text, and the anchors are asserted below: rewording the baseline
# without updating the variants fails at import instead of silently shipping
# a variant that no longer differs from it.
#
# Nothing here ships without a gold measurement — two reasoned prompt changes
# have already measured worse (see PROJECT.md). The weak findings these aim
# at, from the fused labels: Synovitis 0.792, Lateral OA and PF OA next.

_SILENCE_ANCHOR = "Most unmentioned findings land between 5 and 35."

_CORRELATES_BLOCK = _SILENCE_ANCHOR + """
  Co-occurrence is your best evidence for silent findings, and these
  associations are strong in knee MRI:
  - Synovitis is rarely dictated but usually accompanies a joint effusion or
    active osteoarthritis. An effusion or advanced OA with no comment on the
    synovium deserves a clearly raised estimate, not a floor value.
  - Osteoarthritis travels across compartments: described medial OA raises
    the probability of unmentioned patellofemoral and lateral OA.
  - Bone contusion accompanies acute injury (an ACL rupture, a recent
    trauma); a purely chronic degenerative report lowers it.
  - Effusion is raised by any acute internal derangement or active OA.
  A correlate never overrides an explicit statement: a structure described
  as normal stays low regardless of the rest of the report."""

_ROUNDING_ANCHOR = "Do not round to 0, 50, or 100."

_GRANULAR_BLOCK = """Use precise integers — 7, 23, 41, 88 — never multiples
of 5 or 10, and avoid giving two findings the same number unless the report
truly says the same about both. The scores are used to RANK studies, so ties
are wasted answers. Do not round to 0, 50, or 100."""

_RESPOND_ANCHOR = "Respond with ONLY a JSON object mapping each finding to a number:"

_REASON_FIRST_BLOCK = """First write two or three sentences (plain prose, no
braces) sorting the findings: which the report describes as present, which it
calls absent or normal, which it hedges on, and which it never mentions.
Then output the JSON object mapping each finding to a number:"""

_VARIANT_EDITS = {
    # One idea each: named co-occurrence priors for silent findings;
    # pressure against tie blocks; a short sorting step before the numbers.
    "correlates": (_SILENCE_ANCHOR, _CORRELATES_BLOCK),
    "granular": (_ROUNDING_ANCHOR, _GRANULAR_BLOCK),
    "reason_first": (_RESPOND_ANCHOR, _REASON_FIRST_BLOCK),
}

PROBABILITY_PROMPT_VARIANTS = {"baseline": PROBABILITY_PROMPT}
for _name, (_anchor, _replacement) in _VARIANT_EDITS.items():
    assert _anchor in PROBABILITY_PROMPT, f"anchor for {_name!r} lost"
    PROBABILITY_PROMPT_VARIANTS[_name] = PROBABILITY_PROMPT.replace(
        _anchor, _replacement)
    assert PROBABILITY_PROMPT_VARIANTS[_name] != PROBABILITY_PROMPT


def build_probability_prompt(report: str, variant: str = "baseline") -> str:
    """The probability-valued variant of the extraction prompt."""
    definitions = "\n".join(
        f'  "{name}": {definition}' for name, definition in FINDING_DEFINITIONS.items()
    )
    template = PROBABILITY_PROMPT_VARIANTS[variant]
    return template.format(definitions=definitions, report=report)


def parse_probabilities(text: str) -> Dict[str, Optional[float]]:
    """Parse a probability response into {finding: 0..1} or None per finding.

    A missing or unparseable finding stays None rather than defaulting: a
    fabricated 0.5 would sit in the middle of the ranking and is worse than an
    honest gap the caller can fill deliberately.
    """
    result = {name: None for name in FINDING_DEFINITIONS}
    if not isinstance(text, str) or not text.strip():
        return result

    block = _extract_json_block(text)
    if block is None:
        return result

    try:
        payload = json.loads(block)
    except json.JSONDecodeError:
        try:
            payload = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except json.JSONDecodeError:
            return result

    if not isinstance(payload, dict):
        return result

    for name in FINDING_DEFINITIONS:
        value = payload.get(name)
        if isinstance(value, bool) or not isinstance(value, (int, float, str)):
            continue
        try:
            number = float(str(value).strip().rstrip("%"))
        except ValueError:
            continue
        if not 0.0 <= number <= 100.0:
            continue
        result[name] = number / 100.0

    return result


def aggregate_probabilities(samples: List[Dict[str, Optional[float]]]) -> Dict[str, Dict]:
    """Average self-consistency samples, keeping their spread as a weight.

    Agreement across samples is evidence the estimate is stable; disagreement
    is a reason to weight that target down during training.
    """
    aggregated = {}
    for name in FINDING_DEFINITIONS:
        values = [s[name] for s in samples if s.get(name) is not None]
        if not values:
            aggregated[name] = {"probability": None, "confidence": 0.0}
            continue

        mean = sum(values) / len(values)
        if len(values) > 1:
            variance = sum((v - mean) ** 2 for v in values) / len(values)
            spread = variance ** 0.5
        else:
            spread = 0.0
        aggregated[name] = {
            "probability": mean,
            # Full weight at perfect agreement, falling as the samples diverge.
            "confidence": max(0.0, 1.0 - 2.0 * spread),
        }
    return aggregated

## 5. Load the model

`n=3` in the sampling parameters gives three independent samples per report in a
single pass. Majority vote decides each finding and the vote fraction becomes
the confidence weight used later during fusion — findings the model disagrees
with itself about should carry less weight.

In [ ]:
from transformers import AutoTokenizer

MAX_REPORT_CHARS = 6000   # longest report in the corpus is ~4,750
N_SAMPLES = 3
MAX_NEW_TOKENS = 512      # a probability JSON is short; the reason_first
                          # variant spends ~100 extra tokens of prose first

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# vLLM is worth it for 4,349 reports and overkill for the 58-report gold
# checkpoint -- which is the loop we want to run repeatedly while iterating the
# prompt. Its compiled extension is also tied to a specific CUDA runtime and
# breaks whenever Colab's torch shifts under it, so a transformers fallback
# keeps the fast loop available even when vLLM will not import.
USE_VLLM = True
try:
    from vllm import LLM, SamplingParams
except Exception as exc:  # noqa: BLE001
    print("vLLM unavailable, falling back to transformers:", str(exc)[:120])
    USE_VLLM = False

if USE_VLLM:
    # dtype="auto" reads the checkpoint config: AWQ models declare float16,
    # which their kernels require; full-precision Qwen declares bfloat16.
    llm = LLM(model=MODEL, dtype="auto", gpu_memory_utilization=0.90,
              max_model_len=6144, trust_remote_code=True)
else:
    import torch
    from transformers import AutoModelForCausalLM
    if "AWQ" in MODEL:
        # transformers needs the autoawq kernels for AWQ checkpoints; vLLM
        # ships its own, so this only matters on the fallback path.
        import importlib.util, subprocess, sys
        if importlib.util.find_spec("awq") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "autoawq"], check=False)
    dtype = torch.float16 if "AWQ" in MODEL else torch.bfloat16
    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=dtype, device_map="auto")
    hf_model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"


def to_chat(report, variant="baseline"):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",
         "content": build_probability_prompt(str(report)[:MAX_REPORT_CHARS],
                                             variant=variant)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)


def _generate_hf(prompts, n_samples, batch_size=8):
    import torch
    from tqdm.auto import tqdm

    texts = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="generating"):
        chunk = prompts[start:start + batch_size]
        encoded = tokenizer(chunk, return_tensors="pt", padding=True,
                            truncation=True, max_length=6144).to(hf_model.device)
        with torch.no_grad():
            out = hf_model.generate(
                **encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                temperature=0.7, top_p=0.9, num_return_sequences=n_samples,
                pad_token_id=tokenizer.pad_token_id)
        # Strip the prompt so only the completion is parsed.
        generated = out[:, encoded["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for i in range(len(chunk)):
            texts.append(decoded[i * n_samples:(i + 1) * n_samples])
    return texts


def run_extraction(reports, desc="", variant="baseline", n_samples=N_SAMPLES):
    prompts = [to_chat(r, variant) for r in reports]
    print(f"generating {len(prompts)} x {n_samples} samples {desc} "
          f"variant={variant} ({'vLLM' if USE_VLLM else 'transformers'})...")

    if USE_VLLM:
        sampling = SamplingParams(temperature=0.7, top_p=0.9,
                                  max_tokens=MAX_NEW_TOKENS, n=n_samples)
        outputs = llm.generate(prompts, sampling)
        per_report = [[o.text for o in output.outputs] for output in outputs]
    else:
        per_report = _generate_hf(prompts, n_samples)

    return [aggregate_probabilities([parse_probabilities(t) for t in texts])
            for texts in per_report]

## 6. Checkpoint — the gold sweep (Phase C, extraction v3)

**Stop here and read the tables.** This is the only honest measurement
available, and it is deliberately a sweep: five configurations over the 58
gold reports — the shipped baseline at 3 and at 5 self-consistency samples,
then three one-idea prompt variants at 5 samples (`correlates`: named
co-occurrence priors for silent findings, aimed at Synovitis/Lateral/PF OA;
`granular`: pressure against tie blocks; `reason_first`: a short sorting step
before the numbers). ~290 prompts × 5 samples — minutes on an A100.

The raw AUC printed here is a screen, not the gate. **Download every
`llm_probs_gold__*.parquet` this cell produces**, drop them into
`data/meta/gold_sweep/`, and run `python3 tools/score_prompt_variants.py`
locally — that scores the full fusion (the number comparable to the shipped
labels' 0.891) and applies the gate: beat 0.891 by more than 0.005 with no
finding below 0.75. Only a gate-passing variant earns the corpus run below.

In [ ]:
import numpy as np

SWEEP = [("baseline", 3), ("baseline", 5), ("correlates", 5),
         ("granular", 5), ("reason_first", 5)]

sweep_results = {}
for variant, n in SWEEP:
    sweep_results[(variant, n)] = run_extraction(
        gold["Report"].tolist(), "(gold 58)", variant=variant, n_samples=n)

gold_results = sweep_results[("baseline", 3)]


def auc(y_true, scores):
    y_true = np.asarray(y_true); scores = np.asarray(scores, dtype=float)
    pos = y_true == 1
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    unique, inverse, counts = np.unique(scores, return_inverse=True,
                                        return_counts=True)
    sums = np.zeros(len(unique)); np.add.at(sums, inverse, ranks)
    ranks = (sums / counts)[inverse]
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


# Raw per-finding AUC, one column per sweep configuration. This is a screen:
# the decision number is the fused score, computed locally from the dumps by
# tools/score_prompt_variants.py. The shipped baseline's raw macro was 0.860.
def raw_scores(results, name):
    return np.array([r[name]["probability"] if r[name]["probability"] is not None
                     else 0.0 for r in results])


header = "  ".join(f"{v[:9]}@{n}" for v, n in SWEEP)
print(f"\n{'finding':18s} {header}")
print("-" * (20 + 13 * len(SWEEP)))
macros = {key: [] for key in sweep_results}
for name in LABELS:
    truth = gold[name].astype(int).to_numpy()
    row = []
    for key in ((v, n) for v, n in SWEEP):
        a = auc(truth, raw_scores(sweep_results[key], name))
        macros[key].append(a)
        row.append(f"{a:11.3f}")
    print(f"{name:18s} {'  '.join(row)}")
print("-" * (20 + 13 * len(SWEEP)))
print(f"{'macro':18s} " + "  ".join(
    f"{np.nanmean(macros[(v, n)]):11.3f}" for v, n in SWEEP))
print("  shipped baseline raw macro: 0.860 -- fused labels: 0.891 (the gate)")

# Dump every configuration so fusion is scored locally in seconds rather than
# by re-running the model. 58 rows each -- the downloads are instant.
for (variant, n), results in sweep_results.items():
    dump = pd.DataFrame(
        [{name: r[name]["probability"] for name in LABELS} for r in results])
    dump.insert(0, "StudyInstanceUID", gold["StudyInstanceUID"].values)
    for name in LABELS:
        dump[f"{name}_confidence"] = [r[name]["confidence"] for r in results]
    path = f"llm_probs_gold__{variant}_n{n}.parquet"
    dump.to_parquet(path, index=False)
    print("wrote", path)
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        pass

## 7. Full corpus

Only run this after `tools/score_prompt_variants.py` has picked a winner
**locally, on the fused score** — set the two constants to the winning
configuration first. If nothing passes the gate (fused macro > 0.891 + 0.005,
no finding below 0.75), stop here: the sweep cost minutes, and a corpus run
that ships nothing costs an hour. This is the long cell (~45–75 min at 3
samples, scale accordingly).

The gold 58 are re-run at the chosen configuration so the pair of files
section 8 writes is internally consistent.

In [ ]:
import time

# The sweep's verdict goes here, and stays at the shipped default until a
# variant passes the local fused gate.
PROMPT_VARIANT = "baseline"
N_SAMPLES_CORPUS = 3

start = time.time()
gold_results = run_extraction(gold["Report"].tolist(), "(gold 58)",
                              variant=PROMPT_VARIANT,
                              n_samples=N_SAMPLES_CORPUS)
mined = run_extraction(unlabelled["Report"].tolist(), "(4,349 reports)",
                       variant=PROMPT_VARIANT, n_samples=N_SAMPLES_CORPUS)
print(f"done in {(time.time() - start) / 60:.1f} min")

## 8. Save and download

One row per study: `<finding>` (1/0/NaN), `<finding>_severity`,
`<finding>_confidence`, and the evidence span. The evidence is what makes a
disagreement between the two extractors auditable instead of mysterious, so it
is kept rather than discarded.

In [ ]:
def to_frame(results, ids):
    rows = []
    for result in results:
        row = {}
        for name in LABELS:
            entry = result[name]
            row[name] = entry["probability"]
            row[f"{name}_confidence"] = entry.get("confidence")
        rows.append(row)
    frame = pd.DataFrame(rows)
    frame.insert(0, "StudyInstanceUID", list(ids))
    return frame


mined_frame = to_frame(mined, unlabelled["StudyInstanceUID"])
gold_frame = to_frame(gold_results, gold["StudyInstanceUID"])

mined_frame.to_parquet("llm_probs.parquet", index=False)
gold_frame.to_parquet("llm_probs_gold.parquet", index=False)

print(mined_frame.shape)
print("\nprobability distribution per finding (mined):")
for name in LABELS:
    values = mined_frame[name].dropna()
    print(f"  {name:18s} mean {values.mean():5.2f}  "
          f"distinct {values.nunique():5d}  missing {mined_frame[name].isna().mean():5.1%}")

from google.colab import files
files.download("llm_probs.parquet")
files.download("llm_probs_gold.parquet")

## Next

Both parquet files go into `data/meta/`. Fusion then combines these with the
rule-based extractor: agreement becomes a hard target at full weight,
disagreement becomes a soft target at reduced weight, and silence scores zero
with a low sample weight rather than a fabricated prior.